# Pose Correction — Single Frame (GNN)

This notebook trains a simple **graph neural network** to predict per-joint displacement from a **single pose frame**.

- **Dataset**: `Data/output_displacement_single_frame/training_data_displacement_per_frame.npz`
- **Graph**: 12 pose landmarks as nodes with a fixed skeleton adjacency
- **Node features**: `(x, y)` plus the workout-class one-hot (broadcast to every node)
- **Target**: per-node displacement `(dx, dy)` (flattened to 24 dims)
- **Training loss**: masked **MSE** on **displaced joints only** (nodes where \(\|y_{\text{true}}\|_2 > 10^{-6}\)); non-displaced nodes do not contribute to the loss.

## Why does this converge so fast vs `AI/pose_correction/*`?

Compared to the sequence notebooks in `AI/pose_correction/`, this setup can converge much faster because:

- It’s **single-frame** (`sequence_length = 1`), so no temporal model is needed.
- Many rows can be **near-zero displacement** targets, making the loss easy to minimize quickly.
- The split is at the **frame-row level** (not video-level), so train/test can contain frames from the same video, inflating apparent generalization. A fair comparison needs a **video-level split** (requires row→video metadata).


## How to read the results (important)

The dataset has these properties that make the metrics easy to misread:

- **Tiny target magnitude**: each displaced joint coord is `~Uniform(-0.1, +0.1)`.
- **Per row, only 0–3 of 12 joints are displaced** (≈36% of rows are *fully* clean).
- **The unmasked global MAE is dominated by the many zero targets**, so a "predict 0 everywhere" baseline already gives a very small unmasked MAE. That's the trap that makes it look like nothing is converging.

The right number to track is the **masked MAE on displaced joints**, with these reference values:

| Quantity | Value |
|---|---|
| Predict-zero masked-MAE (baseline) | **≈ 0.100** |
| Fixed GraphSAGE-style GNN, ~12 epochs | drops to ~0.048 (half the baseline) |

### Why the GNN was originally stuck at the predict-zero plateau

The earlier plain-GCN version (`SimpleGCN`) had three issues that together collapsed it to predicting zero:

1. **GCN aggregation with self-loops + symmetric norm** mixes a joint with its skeleton neighbors. Since the displacement signal is *per-joint*, this is a low-pass filter that erases exactly what we want to predict.
2. **`LayerNormalization` inside every GCN layer** removes magnitude information at each step. With a target that is small (`~0.05`), this is fatal.
3. **A `LayerNormalization` on the input** mixed pose coordinates with class one-hots, blurring the actual pose signal.

### Fix (applied below)

- **`SAGEConv`**: separate `W_self` for the node's own features and `W_nbr` for the mean of *non-self* neighbors. Self-info is preserved verbatim.
- **No inner LayerNorm/dropout**: keep magnitudes alive.
- **Per-node joint-id one-hot** added to node features so each joint stays identifiable after aggregation.
- **Standardize only the pose channels** (don't touch the class / joint-id one-hots).


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version: {tf.__version__}")
print(f"Keras version: {keras.__version__}")


In [ ]:
data_path = Path("../../Data/output_displacement_single_frame/training_data_displacement_per_frame.npz")
metadata_path = Path("../../Data/output_displacement_single_frame/training_data_displacement_per_frame_metadata.json")

with open(metadata_path, "r") as f:
    metadata = json.load(f)

data = np.load(data_path, allow_pickle=True)
X = data["X"].astype(np.float32)
y = data["y"].astype(np.float32)

pose_dim = int(metadata["X_pose_flat_dim"])  # 12 * n_coords
class_dim = int(metadata["X_class_dim"])     # onehot length
n_landmarks = int(metadata["n_landmarks"])
n_coords = int(metadata["n_coords"])

assert n_landmarks == 12
assert n_coords in (2, 3)
assert X.shape[1] == pose_dim + class_dim
assert y.shape[1] == pose_dim

X_pose = X[:, :pose_dim]
X_class = X[:, pose_dim:]

Y = y

print(f"X_pose: {X_pose.shape}  X_class: {X_class.shape}  Y: {Y.shape}")

In [ ]:
# --------
# Sanity tests + dataset visuals
# --------

y_class = data["y_class"].astype(np.int64)
y_class_onehot = data["y_class_onehot"].astype(np.float32)
class_names = [str(x) for x in data["class_names"].tolist()]

assert X.ndim == 2 and y.ndim == 2
assert X.shape[0] == y.shape[0]
assert y_class.shape[0] == X.shape[0]
assert y_class_onehot.shape == (X.shape[0], len(class_names))
assert np.all((y_class_onehot.sum(axis=1) == 1.0) | (y_class_onehot.sum(axis=1) == 0.0))

print("Basic checks:")
print("  any NaNs in X?", bool(np.isnan(X).any()))
print("  any NaNs in y?", bool(np.isnan(y).any()))
print("  class_names:", len(class_names))

# How much of the dataset is 'no displacement'? (y ~= 0)
y_norm = np.linalg.norm(y, axis=1)
print("\nTarget magnitude stats (L2 over 24 dims):")
print("  mean:", float(np.mean(y_norm)))
print("  p50:", float(np.percentile(y_norm, 50)))
print("  p90:", float(np.percentile(y_norm, 90)))
print("  frac(|y|<1e-6):", float(np.mean(y_norm < 1e-6)))
print("  frac(|y|<1e-3):", float(np.mean(y_norm < 1e-3)))

plt.figure(figsize=(7, 4))
plt.hist(y_norm, bins=80)
plt.title("Target displacement magnitude (L2) histogram")
plt.xlabel("||y||")
plt.ylabel("count")
plt.tight_layout()
plt.show()


In [ ]:
# (N, 12*C) -> (N, 12, C)
X_pose_nodes = X_pose.reshape((-1, n_landmarks, n_coords))
Y_nodes = Y.reshape((-1, n_landmarks, n_coords))

# Broadcast class one-hot to each node: (N, 17) -> (N, 12, 17)
X_class_nodes = np.repeat(X_class[:, None, :], repeats=n_landmarks, axis=1)

# Per-node joint-ID one-hot — this is critical so the GNN can preserve
# 'which joint is which' even after neighbor aggregation.
_joint_id_eye = np.eye(n_landmarks, dtype=np.float32)
X_id_nodes = np.broadcast_to(_joint_id_eye[None, :, :], (X.shape[0], n_landmarks, n_landmarks)).copy()

# Node features: (x, y[, z]) + class one-hot + joint-id one-hot
X_nodes = np.concatenate([X_pose_nodes, X_class_nodes, X_id_nodes], axis=-1).astype(np.float32)

print(f"X_nodes: {X_nodes.shape} (node_feat_dim={X_nodes.shape[-1]})")
print(f"Y_nodes: {Y_nodes.shape}")


In [ ]:
# Landmark order (from Data/video_pose_extractor_displacement_per_frame.py):
# 0 left_shoulder, 1 right_shoulder, 2 left_elbow, 3 right_elbow,
# 4 left_wrist, 5 right_wrist, 6 left_hip, 7 right_hip,
# 8 left_knee, 9 right_knee, 10 left_ankle, 11 right_ankle

edges = [
    (0, 1),  # shoulders
    (6, 7),  # hips
    (0, 2), (2, 4),  # left arm
    (1, 3), (3, 5),  # right arm
    (0, 6), (1, 7),  # torso
    (6, 8), (8, 10),  # left leg
    (7, 9), (9, 11),  # right leg
]

A = np.zeros((n_landmarks, n_landmarks), dtype=np.float32)
for i, j in edges:
    A[i, j] = 1.0
    A[j, i] = 1.0

# NOTE: We deliberately *do not* add self-loops here.
# Our SAGE-style layer applies a separate learned weight to the node's
# own features (W_self * x) and to the neighbor mean (W_nbr * AGG(x_neighbors)).
# Mixing self-loops into A would defeat that separation and the model
# would collapse toward predicting zero (low-pass smoothing kills the
# per-joint anomaly signal we are trying to learn).

# Row-normalized adjacency = mean over neighbors.
deg = np.sum(A, axis=1)
A_norm = (A / np.maximum(deg[:, None], 1.0)).astype(np.float32)

A_norm_tf = tf.constant(A_norm)
print("A_norm shape:", A_norm.shape)


In [ ]:
# NOTE: The dataset rows are per-frame. A truly fair split is by *video*,
# but the row->video mapping is not present in the .npz by default.
# We at least split stratified by workout class to avoid class imbalance.

def stratified_split_indices(y_class_idx: np.ndarray, *, seed: int = 42, train_frac=0.8, val_frac=0.1):
    rng = np.random.default_rng(seed)
    train, val, test = [], [], []
    for c in np.unique(y_class_idx):
        c_idx = np.where(y_class_idx == c)[0]
        rng.shuffle(c_idx)
        n = len(c_idx)
        n_train = int(train_frac * n)
        n_val = int(val_frac * n)
        train.append(c_idx[:n_train])
        val.append(c_idx[n_train : n_train + n_val])
        test.append(c_idx[n_train + n_val :])
    return np.concatenate(train), np.concatenate(val), np.concatenate(test)

train_idx, val_idx, test_idx = stratified_split_indices(y_class, seed=42)

rng = np.random.default_rng(42)
rng.shuffle(train_idx)
rng.shuffle(val_idx)
rng.shuffle(test_idx)

X_train, y_train = X_nodes[train_idx], Y_nodes[train_idx]
X_val, y_val = X_nodes[val_idx], Y_nodes[val_idx]
X_test, y_test = X_nodes[test_idx], Y_nodes[test_idx]

print("Split sizes:")
print(f"  train: {X_train.shape}")
print(f"  val:   {X_val.shape}")
print(f"  test:  {X_test.shape}")

for name, idxs in [("train", train_idx), ("val", val_idx), ("test", test_idx)]:
    counts = np.bincount(y_class[idxs], minlength=len(class_names))
    top = np.argsort(-counts)[:5]
    print(f"\n{name} top-5 classes:")
    for k in top:
        if counts[k] > 0:
            print(f"  {class_names[k]:<22} {int(counts[k])}")


In [ ]:
# --------
# Non-zero target ratio (and optional sample weighting)
# --------

eps = 1e-6
train_norm = np.linalg.norm(y_train.reshape((y_train.shape[0], -1)), axis=1)
val_norm = np.linalg.norm(y_val.reshape((y_val.shape[0], -1)), axis=1)
test_norm = np.linalg.norm(y_test.reshape((y_test.shape[0], -1)), axis=1)

print("Non-zero target ratios:")
print(f"  train frac(|y|<{eps}): {float(np.mean(train_norm < eps)):.3f}")
print(f"  val   frac(|y|<{eps}): {float(np.mean(val_norm < eps)):.3f}")
print(f"  test  frac(|y|<{eps}): {float(np.mean(test_norm < eps)):.3f}")

nonzero_weight = 5.0
sample_weight_train = np.where(train_norm < eps, 1.0, nonzero_weight).astype(np.float32)
sample_weight_val = np.where(val_norm < eps, 1.0, nonzero_weight).astype(np.float32)

print("Sample weights:")
print("  train mean weight:", float(sample_weight_train.mean()))
print("  val   mean weight:", float(sample_weight_val.mean()))


In [ ]:
# GraphSAGE-style layer: keep self-features and neighbor-aggregated features
# in *separate* learnable transforms, then sum.  This avoids the over-smoothing
# pathology of plain GCN where W * (A_hat @ x) (with self-loops) blends a
# joint with its neighbors and erases the per-joint anomaly signal.
class SAGEConv(layers.Layer):
    def __init__(self, units: int, activation="relu", **kwargs):
        super().__init__(**kwargs)
        self.units = int(units)
        self.activation = keras.activations.get(activation)
        self.lin_self = layers.Dense(self.units, use_bias=True)
        self.lin_nbr = layers.Dense(self.units, use_bias=True)

    def call(self, x, training=False):
        nbr = tf.linalg.matmul(A_norm_tf, x)  # mean over neighbors (no self-loops)
        h = self.lin_self(x) + self.lin_nbr(nbr)
        return self.activation(h)


# Standardize ONLY the pose part of the node features; leave the class +
# joint-id one-hots untouched.  Mixing them through a single LayerNorm
# was wiping out the small-magnitude pose signal we actually need.
_pose_mean = X_train[..., :n_coords].mean(axis=(0, 1), keepdims=True).astype(np.float32)
_pose_std = X_train[..., :n_coords].std(axis=(0, 1), keepdims=True).astype(np.float32) + 1e-6


def _standardize_pose_in_nodes(arr: np.ndarray) -> np.ndarray:
    out = arr.copy()
    out[..., :n_coords] = (arr[..., :n_coords] - _pose_mean) / _pose_std
    return out


X_train = _standardize_pose_in_nodes(X_train)
X_val = _standardize_pose_in_nodes(X_val)
X_test = _standardize_pose_in_nodes(X_test)

node_feat_dim = int(X_train.shape[-1])

inp = keras.Input(shape=(n_landmarks, node_feat_dim), name="X_nodes")
x = SAGEConv(128)(inp)
x = SAGEConv(128)(x)
x = SAGEConv(64)(x)

x = layers.Dense(64, activation="relu")(x)
out = layers.Dense(n_coords, name="y_nodes_pred")(x)

model = keras.Model(inp, out)

# Loss / metrics: only joints with non-zero *ground-truth* displacement (per sample).
_DISP_EPS = 1e-6


def masked_displaced_joint_mse(y_true, y_pred):
    joint_norm_true = tf.linalg.norm(y_true, axis=-1)
    mask = tf.cast(joint_norm_true > _DISP_EPS, tf.float32)
    mask_w = tf.expand_dims(mask, -1)
    se = tf.square(y_true - y_pred) * mask_w
    denom = tf.reduce_sum(mask_w) + 1e-8
    return tf.reduce_sum(se) / denom


def masked_displaced_joint_mae(y_true, y_pred):
    joint_norm_true = tf.linalg.norm(y_true, axis=-1)
    mask = tf.cast(joint_norm_true > _DISP_EPS, tf.float32)
    mask_w = tf.expand_dims(mask, -1)
    ae = tf.abs(y_true - y_pred) * mask_w
    denom = tf.reduce_sum(mask_w) + 1e-8
    return tf.reduce_sum(ae) / denom


model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=masked_displaced_joint_mse,
    metrics=[masked_displaced_joint_mae],
)
model.summary()


In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=25, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=15, min_lr=1e-6),
]

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=250,
    batch_size=256,
    callbacks=callbacks,
    verbose=1,
)


In [ ]:
# --------
# Baselines (to ensure we're solving the same regression problem)
# --------

def mae_global_nodes(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.mean(np.abs(y_true - y_pred)))


def mae_masked_displaced(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    yt = y_true.reshape((-1, n_landmarks, n_coords))
    yp = y_pred.reshape((-1, n_landmarks, n_coords))
    mask = (np.linalg.norm(yt, axis=-1) > 1e-6)[..., None]
    ae = np.abs(yt - yp) * mask
    return float(ae.sum() / max(mask.sum(), 1.0))


zero_pred = np.zeros_like(y_test)
mean_node = np.mean(y_train, axis=0, keepdims=True)
mean_pred = np.repeat(mean_node, repeats=y_test.shape[0], axis=0)

print("Baseline: predict all zeros")
print(f"  unmasked MAE         : {mae_global_nodes(y_test, zero_pred):.6f}  (dominated by zero targets, not informative)")
print(f"  masked MAE (disp jt) : {mae_masked_displaced(y_test, zero_pred):.6f}  <- track THIS")

print("\nBaseline: predict train mean")
print(f"  unmasked MAE         : {mae_global_nodes(y_test, mean_pred):.6f}")
print(f"  masked MAE (disp jt) : {mae_masked_displaced(y_test, mean_pred):.6f}")


In [ ]:
def r2_score_np(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_true = np.asarray(y_true, dtype=np.float32).reshape((y_true.shape[0], -1))
    y_pred = np.asarray(y_pred, dtype=np.float32).reshape((y_pred.shape[0], -1))
    ss_res = float(np.sum((y_true - y_pred) ** 2))
    ss_tot = float(np.sum((y_true - np.mean(y_true, axis=0, keepdims=True)) ** 2))
    return 1.0 - ss_res / (ss_tot + 1e-12)

y_pred = model.predict(X_test, batch_size=1024)

mae = float(np.mean(np.abs(y_test - y_pred)))
r2 = r2_score_np(y_test, y_pred)

print(f"Test MAE (mean over all nodes/dims): {mae:.6f}")
print(f"Test R2 (global): {r2:.6f}")


In [ ]:
out_dir = Path("./saved_models")
out_dir.mkdir(parents=True, exist_ok=True)
save_path = out_dir / "gcn_single_frame_displacement.keras"
model.save(save_path)
print(f"Saved to: {save_path}")


In [ ]:
# --------
# Training curves
# --------

plt.figure(figsize=(7, 4))
plt.plot(history.history.get("loss", []), label="train")
plt.plot(history.history.get("val_loss", []), label="val")
plt.title("Loss (masked MSE, displaced joints)")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
_mae_key = "masked_displaced_joint_mae" if "masked_displaced_joint_mae" in history.history else "mae"
plt.plot(history.history.get(_mae_key, []), label="train")
plt.plot(history.history.get(f"val_{_mae_key}", []), label="val")
plt.title("MAE (displaced joints only)")
plt.xlabel("epoch")
plt.ylabel("MAE")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# # Global error histogram
# err = (y_test - y_pred)
# err_norm = np.linalg.norm(err, axis=1)
# plt.figure(figsize=(7, 4))
# plt.hist(err_norm, bins=80)
# plt.title("Test error magnitude (L2) histogram")
# plt.xlabel("||y_true - y_pred||")
# plt.ylabel("count")
# plt.tight_layout()
# plt.show()

# # Scatter: true vs pred norms
# plt.figure(figsize=(5, 5))
# plt.scatter(np.linalg.norm(y_test, axis=1), np.linalg.norm(y_pred, axis=1), s=3, alpha=0.25)
# plt.title("||y||: true vs predicted")
# plt.xlabel("||y_true||")
# plt.ylabel("||y_pred||")
# plt.tight_layout()
# plt.show()

In [ ]:
# --------
# Prediction visualizations + qualitative skeleton plots
# --------

# Reuse edges defined above in this notebook

def plot_skeleton_with_arrows(pose_xy, disp_true_xy, disp_pred_xy, title=""):
    from matplotlib.lines import Line2D

    pose_xy = np.asarray(pose_xy)
    dt = np.asarray(disp_true_xy)
    dp = np.asarray(disp_pred_xy)

    fig, ax = plt.subplots(figsize=(6, 6))

    for i, j in edges:
        ax.plot([pose_xy[i, 0], pose_xy[j, 0]], [pose_xy[i, 1], pose_xy[j, 1]], "k-", alpha=0.35, zorder=1)
    ax.scatter(pose_xy[:, 0], pose_xy[:, 1], s=25, zorder=2)

    # Draw BOTH: true first, pred on top.
    ax.quiver(
        pose_xy[:, 0],
        pose_xy[:, 1],
        dt[:, 0],
        dt[:, 1],
        angles="xy",
        scale_units="xy",
        scale=1.0,
        color="tab:green",
        alpha=0.85,
        width=0.004,
        zorder=3,
    )
    ax.quiver(
        pose_xy[:, 0],
        pose_xy[:, 1],
        dp[:, 0],
        dp[:, 1],
        angles="xy",
        scale_units="xy",
        scale=1.0,
        color="tab:red",
        alpha=0.85,
        width=0.004,
        zorder=4,
    )

    proxies = [
        Line2D([0], [0], color="tab:green", lw=2, label="true displacement"),
        Line2D([0], [0], color="tab:red", lw=2, label="pred displacement"),
    ]
    ax.legend(handles=proxies, loc="upper right")

    true_ma = float(np.mean(np.abs(dt)))
    pred_ma = float(np.mean(np.abs(dp)))
    err_ma = float(np.mean(np.abs(dt - dp)))
    ax.text(
        0.02,
        0.02,
        f"mean|true|={true_ma:.4f}\nmean|pred|={pred_ma:.4f}\nmean|err|={err_ma:.4f}",
        transform=ax.transAxes,
        fontsize=10,
        va="bottom",
        ha="left",
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.75, edgecolor="none"),
        zorder=10,
    )

    ax.invert_yaxis()
    ax.set_title(title)
    ax.set_aspect("equal", adjustable="box")
    fig.tight_layout()
    plt.show()

err = (y_test - y_pred)
err_flat = err.reshape((err.shape[0], -1))
err_norm = np.linalg.norm(err_flat, axis=1)

plt.figure(figsize=(7, 4))
plt.hist(err_norm, bins=80)
plt.title("Test error magnitude (L2) histogram")
plt.xlabel("||y_true - y_pred||")
plt.ylabel("count")
plt.tight_layout()
plt.show()

plt.figure(figsize=(5, 5))
plt.scatter(np.linalg.norm(y_test.reshape((y_test.shape[0], -1)), axis=1), np.linalg.norm(y_pred.reshape((y_pred.shape[0], -1)), axis=1), s=3, alpha=0.25)
plt.title("||y||: true vs predicted")
plt.xlabel("||y_true||")
plt.ylabel("||y_pred||")
plt.tight_layout()
plt.show()

per_joint_mae = np.mean(np.abs(err), axis=(0, 2))
plt.figure(figsize=(9, 3))
plt.bar(np.arange(n_landmarks), per_joint_mae)
plt.title("Per-joint MAE (averaged over x/y)")
plt.xlabel("joint index")
plt.ylabel("MAE")
plt.tight_layout()
plt.show()

# Qualitative samples
rng = np.random.default_rng(0)
for k in rng.choice(np.arange(X_test.shape[0]), size=3, replace=False):
    pose_xy = X_test[k, :, :n_coords][:, :2]
    y_true_xy = y_test[k, :, :2]
    y_pred_xy = y_pred[k, :, :2]
    plot_skeleton_with_arrows(pose_xy, y_true_xy, y_pred_xy, title=f"sample {int(k)}")
